# Praktikum Mandiri — Machine Learning Prediksi RPM Motor DC → ESP32

**Satu Google Colab, dari CSV mentah sampai ZIP MicroPython siap diunggah ke ESP32.**

**Tujuan:** mempelajari **regresi** (*bukan klasifikasi*) untuk memperkirakan RPM dari getaran accelerometer MPU6050. Encoder hanya sebagai label dan pembanding; **PWM, setpoint dan RPM encoder bukan fitur input ML**.

**Tahapan:** unggah CSV/ZIP → cek kualitas → ekstraksi 36 fitur per window 400 sampel → pembagian *per file rekaman* → latih dan evaluasi Random Forest → latih model final pada seluruh data → konversi model ke RFR2 → uji kesetiaan konversi → unduh ZIP **`main.py` + `model_rf.bin`**.

**Perangkat:** ESP32 DevKit, L298N, motor DC dengan encoder, MPU6050. Kode ESP32 meneruskan konfigurasi pin/PID dari praktikum motor yang sudah bekerja; komunikasi serial setpoint dan output prediksi disediakan di `main.py`.

> **Penting:** notebook ini menargetkan **200 Hz, window 400 sampel (~2 detik), dan 36 fitur**; jangan mengubah definisi fitur tanpa mengubah program ESP32. Metrik pengujian berasal dari model evaluasi; model deployment kemudian dilatih ulang pada seluruh data valid. Waktu inferensi dan penggunaan RAM di ESP32 tetap perlu diukur secara langsung.

## 1 — Pengaturan praktikum

Jalankan sel secara berurutan. Atur `MAX_RPM_STD` jika data RPM sangat berfluktuasi; tidak perlu mengubah jumlah fitur atau parameter window.

In [ ]:
import os, io, sys, json, zipfile, shutil, warnings, struct, base64, zlib, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from google.colab import files as colab_files
    IS_COLAB = True
except ImportError:
    colab_files = None
    IS_COLAB = False

RANDOM_STATE = 42
SAMPLING_HZ = 200          # Konsisten dengan main.py pada ESP32
WINDOW_N = 400            # 2 s @ 200 Hz; jangan ganti tanpa mengubah main.py
HOP_N = 200               # Overlap 50% hanya untuk training laptop
MAX_RPM_STD = 12.0        # None untuk menonaktifkan penolakan window fluktuatif
MIN_RPM = 1.0
FFT_MIN_HZ = 2.0
FFT_MAX_HZ = 95.0
N_TREES = 300
MIN_SAMPLES_LEAF = 2
TEST_FRACTION = 0.25

BASE_DIR = Path('/content') if IS_COLAB else Path.cwd()
DATA_DIR = BASE_DIR / 'motor_mpu_data'
OUTPUT_DIR = BASE_DIR / 'output_rpm_mpu'
for p in [DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
print('Mode:', 'Colab' if IS_COLAB else 'Python lokal')
print('Folder:', DATA_DIR)


## 2 — Unggah dataset awal

Unggah banyak **CSV** hasil logging ESP32 sekaligus atau **satu ZIP yang berisi CSV**. Format minimal: `ax_g`, `ay_g`, `az_g`, `rpm_actual` (atau `rpm_encoder`), serta sebaiknya `t_us`. Satu file = satu rekaman independen. Hindari menggabungkan banyak run dalam satu CSV.

Untuk Google Colab, sel ini membersihkan CSV pada folder kerja khusus sebelum menerima unggahan baru.

In [ ]:
def simpan_data(uploaded, folder):
    count = 0
    for name, blob in uploaded.items():
        if name.lower().endswith('.csv'):
            entries = [(Path(name).name, blob)]
        elif name.lower().endswith('.zip'):
            entries = []
            with zipfile.ZipFile(io.BytesIO(blob)) as z:
                for inf in z.infolist():
                    if inf.is_dir() or not inf.filename.lower().endswith('.csv'):
                        continue
                    if inf.file_size > 30_000_000:
                        raise ValueError('File CSV terlalu besar: ' + inf.filename)
                    entries.append((Path(inf.filename).name, z.read(inf)))
        else:
            print('Dilewati:', name)
            continue
        for filename, content in entries:
            stem = Path(filename).stem
            dest = folder / (stem + '.csv')
            j = 2
            while dest.exists():
                dest = folder / f'{stem}_{j}.csv'
                j += 1
            dest.write_bytes(content)
            count += 1
    return count

if IS_COLAB:
    for p in DATA_DIR.glob('*.csv'):
        p.unlink()
    print('Pilih file CSV atau ZIP dari komputer Anda...')
    uploaded = colab_files.upload()
    print('CSV diterima:', simpan_data(uploaded, DATA_DIR))
else:
    print('Mode lokal: letakkan file CSV di:', DATA_DIR)

csv_paths = sorted(DATA_DIR.glob('*.csv'))
print('Jumlah rekaman:', len(csv_paths))
for p in csv_paths[:30]:
    print(' -', p.name)
if len(csv_paths) < 3:
    raise ValueError('Perlu minimal 3 file rekaman untuk pemisahan training/testing berdasarkan file.')


## 3 — Pemeriksaan data dan kualitas sampling

Label adalah RPM **aktual encoder**, bukan setpoint. Nilai negatif diubah menjadi magnitudo RPM karena eksperimen memperkirakan **kecepatan putaran**. Rekaman yang punya timestamp tidak valid atau kolom yang hilang akan ditolak/ditandai.

In [ ]:
SENSOR_COLS = ['ax_g', 'ay_g', 'az_g']
LABEL_CANDIDATES = ['rpm_actual', 'rpm_encoder', 'rpm_filtered']


def load_motor_csv(filepath, need_label=True):
    df = pd.read_csv(filepath)
    df.columns = [str(c).strip() for c in df.columns]
    if len(set(df.columns)) != len(df.columns):
        raise ValueError(f'{Path(filepath).name}: nama kolom duplikat')
    missing = [c for c in SENSOR_COLS if c not in df]
    if missing:
        raise ValueError(f'{Path(filepath).name}: kolom sensor tidak ditemukan: {missing}')

    label_col = next((c for c in LABEL_CANDIDATES if c in df), None)
    if need_label and label_col is None:
        raise ValueError(f'{Path(filepath).name}: tidak ada rpm_actual / rpm_encoder')

    numeric_cols = SENSOR_COLS.copy()
    if label_col is not None:
        numeric_cols.append(label_col)
    if 'setpoint_rpm' in df:
        numeric_cols.append('setpoint_rpm')
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=SENSOR_COLS + ([label_col] if need_label else [])).copy()
    if df.empty:
        raise ValueError(f'{Path(filepath).name}: tidak ada sampel valid')

    df['rpm_label'] = df[label_col].abs() if label_col else np.nan
    if 'setpoint_rpm' not in df:
        df['setpoint_rpm'] = np.nan

    # Hitung waktu relatif (detik): prefer t_us -> dt_us -> 200 Hz default.
    if 't_us' in df:
        raw_time = pd.to_numeric(df['t_us'], errors='coerce').to_numpy(dtype=float)
        if np.isfinite(raw_time).all() and np.all(np.diff(raw_time) > 0):
            t_s = (raw_time - raw_time[0]) / 1e6
        else:
            t_s = np.arange(len(df)) / FS_FALLBACK
            warnings.warn(f'{Path(filepath).name}: t_us tidak monoton; gunakan {FS_FALLBACK} Hz')
    elif 'dt_us' in df:
        dt = pd.to_numeric(df['dt_us'], errors='coerce').to_numpy(dtype=float)
        dt[~np.isfinite(dt) | (dt <= 0)] = 1e6 / FS_FALLBACK
        t_s = np.cumsum(dt) / 1e6
        t_s -= t_s[0]
    else:
        t_s = np.arange(len(df)) / FS_FALLBACK
        warnings.warn(f'{Path(filepath).name}: tidak ada timestamp; diasumsikan {FS_FALLBACK} Hz')

    df['t_s'] = t_s
    dt_valid = np.diff(t_s)
    dt_valid = dt_valid[dt_valid > 0]
    fs = (1 / np.median(dt_valid)) if len(dt_valid) else FS_FALLBACK
    if not (20 <= fs <= 2000):
        warnings.warn(f'{Path(filepath).name}: frekuensi terdeteksi {fs:.1f} Hz tidak wajar; gunakan {FS_FALLBACK} Hz')
        fs = FS_FALLBACK
    jitter_pct = 100 * np.std(dt_valid) / np.mean(dt_valid) if len(dt_valid) > 1 else np.nan
    return df.reset_index(drop=True), float(fs), label_col, jitter_pct

ringkasan_file = []
for file in csv_paths:
    try:
        df, fs, label, jitter = load_motor_csv(file, need_label=True)
        ringkasan_file.append({
            'file': file.name, 'sampel': len(df), 'fs_estimasi_hz': round(fs, 2),
            'jitter_dt_pct': round(jitter, 2), 'label': label,
            'setpoint_rpm': round(float(df['setpoint_rpm'].median()), 2) if df['setpoint_rpm'].notna().any() else np.nan,
            'rpm_median': round(float(df['rpm_label'].median()), 2),
            'rpm_std': round(float(df['rpm_label'].std()), 2),
        })
    except Exception as e:
        print('GAGAL membaca', file.name, ':', e)

quality = pd.DataFrame(ringkasan_file)
if quality.empty:
    raise RuntimeError('Tidak ada CSV valid. Periksa nama kolom dan isinya.')
display(quality)
if (quality['jitter_dt_pct'] > 10).any():
    print('PERHATIAN: Jitter timestamp > 10% pada sebagian rekaman. FFT dapat kurang akurat.')

## 4 — Ekstraksi fitur getaran (36 fitur)

Setiap window 400 sampel menghasilkan 12 fitur per sumbu X, Y, Z: std, RMS AC, peak, peak-to-peak, kurtosis, dominant frequency, spectral centroid, log spectral energy, dan empat bandpower. Pengolahan **interpolasi timestamp + Hann + FFT** sama definisinya dengan `main.py` di ESP32.

> **Catatan:** notebook hanya menerima rekaman dengan sampling sekitar 200 Hz untuk ekspor ini. Jika data Anda bukan 200 Hz, lakukan akuisisi ulang/ubah seluruh pipeline secara konsisten, bukan hanya variabel di Colab.

In [ ]:
def fitur_satu_sumbu(t, a, fs, nama):
    t = np.asarray(t, dtype=float)
    a = np.asarray(a, dtype=float)
    n = len(a)
    # Sampling aktual dapat memiliki jitter; interpolasi ke grid seragam.
    t_uniform = np.linspace(t[0], t[-1], n)
    a_uniform = np.interp(t_uniform, t, a)
    x = a_uniform - np.mean(a_uniform)  # AC vibration; hilangkan DC/gravitasi
    dur = max(t_uniform[-1] - t_uniform[0], 1e-9)
    fs_uniform = (n - 1) / dur

    std = float(np.std(x))
    rms = float(np.sqrt(np.mean(x ** 2)))
    peak = float(np.max(np.abs(x)))
    ptp = float(np.ptp(x))
    kurtosis = float(np.mean((x / std) ** 4)) if std > 1e-10 else 0.0

    freqs = np.fft.rfftfreq(n, d=1.0 / fs_uniform)
    spectrum = np.abs(np.fft.rfft(x * np.hanning(n))) ** 2
    upper = min(FFT_MAX_HZ, fs_uniform * 0.48)
    valid = (freqs >= FFT_MIN_HZ) & (freqs <= upper)
    p = spectrum[valid]
    f = freqs[valid]
    if len(p) == 0 or np.sum(p) < 1e-15:
        f_dom, centroid, energy = 0.0, 0.0, 0.0
    else:
        f_dom = float(f[np.argmax(p)])
        centroid = float(np.sum(f * p) / np.sum(p))
        energy = float(np.sum(p))

    fitur = {
        f'{nama}_std': std,
        f'{nama}_rms_ac': rms,
        f'{nama}_peak': peak,
        f'{nama}_ptp': ptp,
        f'{nama}_kurtosis': kurtosis,
        f'{nama}_dom_hz': f_dom,
        f'{nama}_centroid_hz': centroid,
        f'{nama}_log_spectral_energy': float(np.log1p(energy)),
    }
    for low, high in [(2, 10), (10, 30), (30, 60), (60, 95)]:
        part = (freqs >= low) & (freqs < min(high, upper + 1e-9))
        fitur[f'{nama}_band_{low}_{high}_log'] = float(np.log1p(np.sum(spectrum[part])))
    return fitur




def extract_windows(filepath, require_label=True):
    df, fs, _, _ = load_motor_csv(filepath, need_label=require_label)
    if not (170.0 <= fs <= 230.0):
        warnings.warn(f'{Path(filepath).name}: sampling {fs:.1f} Hz, di luar target 200 Hz; file dilewati')
        return pd.DataFrame()
    daftar = []
    for start in range(0, len(df) - WINDOW_N + 1, HOP_N):
        frame = df.iloc[start:start + WINDOW_N]
        t = frame['t_s'].to_numpy(dtype=float)
        dt = np.diff(t)
        if (not np.isfinite(t).all() or np.any(dt <= 0)
                or np.max(dt) > 3.0 / fs):
            continue
        fs_window = (WINDOW_N - 1) / (t[-1] - t[0])
        if not (170.0 <= fs_window <= 230.0):
            continue
        rpm = frame['rpm_label'].to_numpy(dtype=float)
        median_rpm = float(np.nanmedian(rpm)) if np.isfinite(rpm).any() else np.nan
        rpm_std = float(np.nanstd(rpm)) if np.isfinite(rpm).any() else np.nan
        if require_label:
            if not np.isfinite(median_rpm) or median_rpm < MIN_RPM:
                continue
            if MAX_RPM_STD is not None and rpm_std > MAX_RPM_STD:
                continue
        row = {
            'recording': Path(filepath).name,
            'window_start_s': float(t[0]),
            'window_end_s': float(t[-1]),
            'setpoint_rpm': float(frame['setpoint_rpm'].median()) if frame['setpoint_rpm'].notna().any() else np.nan,
            'rpm_target': median_rpm,
            'rpm_window_std': rpm_std,
        }
        for col in SENSOR_COLS:
            row.update(fitur_satu_sumbu(t, frame[col].to_numpy(), fs_window, col.split('_')[0]))
        daftar.append(row)
    return pd.DataFrame(daftar)

frames = []
for fp in csv_paths:
    try:
        f = extract_windows(fp)
        print(f'{fp.name}: {len(f)} window valid')
        if len(f):
            frames.append(f)
    except Exception as exc:
        print('Dilewati:', fp.name, exc)
if not frames:
    raise RuntimeError('Tidak ada window valid. Periksa CSV, sampling, dan MAX_RPM_STD.')
features = pd.concat(frames, ignore_index=True)
meta_cols = ['recording','window_start_s','window_end_s','setpoint_rpm','rpm_target','rpm_window_std']
feature_cols = [c for c in features.columns if c not in meta_cols]
if len(feature_cols) != 36:
    raise RuntimeError('Jumlah fitur bukan 36. Sinkronkan notebook dan kode ESP32.')
if not np.isfinite(features[feature_cols].to_numpy()).all():
    raise ValueError('Fitur NaN/Inf; periksa akuisisi.')
if features['recording'].nunique() < 3:
    raise ValueError('Jumlah file dengan window valid <3. Tambah rekaman.')
features.to_csv(OUTPUT_DIR / 'fitur_getaran.csv', index=False)
print('Total window:', len(features), '| file valid:', features['recording'].nunique(), '| fitur:', len(feature_cols))
display(features[['recording','window_start_s','setpoint_rpm','rpm_target','rpm_window_std']].head(15).round(3))


## 5 — Eksplorasi awal: getaran dan RPM

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(features['rpm_target'], features['ax_rms_ac'], alpha=0.7)
axes[0].set(xlabel='RPM encoder (median per window)', ylabel='RMS AC X (g)', title='Getaran vs RPM')
axes[0].grid(alpha=0.3)
axes[1].scatter(features['rpm_target'], features['ax_dom_hz'], alpha=0.7)
axes[1].set(xlabel='RPM encoder (median per window)', ylabel='Frekuensi dominan X (Hz)', title='Frekuensi getaran vs RPM')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Pembagian data per rekaman

Kita pisahkan berdasarkan **nama file**, bukan window acak. Bila ada pengulangan pada minimal dua setpoint, gunakan beberapa rekaman terpisah sebagai data uji. Jika tidak, gunakan *GroupShuffleSplit*. Hasil uji adalah **regresi RPM**.

In [ ]:
files_count = features['recording'].nunique()
if files_count < 3:
    raise ValueError('Butuh minimal 3 rekaman/file terpisah untuk evaluasi awal. Idealnya 3 run tiap RPM.')

file_summary = features.groupby('recording').agg(
    setpoint=('setpoint_rpm','median'), n_windows=('rpm_target','size')
).reset_index()
counts = file_summary.groupby('setpoint')['recording'].nunique()
repeated_rpm = counts[counts >= 2]

if file_summary['setpoint'].notna().all() and len(repeated_rpm) >= 2:
    rng = np.random.default_rng(RANDOM_STATE)
    test_files = []
    for rpm in repeated_rpm.index:
        candidates = file_summary.loc[file_summary['setpoint'] == rpm, 'recording'].to_list()
        test_files.append(str(rng.choice(candidates)))
    test_mask = features['recording'].isin(test_files)
    train_mask = ~test_mask
    split_method = 'Uji pada rekaman terpisah dari RPM yang memiliki pengulangan'
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_STATE)
    tr_idx, te_idx = next(splitter.split(features, groups=features['recording']))
    train_mask = pd.Series(False, index=features.index)
    test_mask = pd.Series(False, index=features.index)
    train_mask.iloc[tr_idx] = True
    test_mask.iloc[te_idx] = True
    split_method = 'GroupShuffleSplit per file (bisa menguji RPM baru yang belum dilatih)'

train_data = features.loc[train_mask].copy()
test_data = features.loc[test_mask].copy()
assert set(train_data['recording']).isdisjoint(set(test_data['recording']))
if train_data['rpm_target'].nunique() < 2:
    raise ValueError('RPM pada train tidak cukup bervariasi. Kumpulkan lebih banyak RPM/run.')

X_train, y_train = train_data[feature_cols], train_data['rpm_target']
X_test, y_test = test_data[feature_cols], test_data['rpm_target']
print('Strategi split:', split_method)
print('Train:', len(X_train), 'window dari', train_data['recording'].nunique(), 'file')
print('Test :', len(X_test), 'window dari', test_data['recording'].nunique(), 'file')
print('File test:', sorted(test_data['recording'].unique().tolist()))
print('Rentang RPM train:', round(y_train.min(),1), '-', round(y_train.max(),1))
print('Rentang RPM test :', round(y_test.min(),1), '-', round(y_test.max(),1))

## 7 — Model evaluasi: Random Forest vs baseline

Model evaluasi **hanya dilatih pada data train**. Tampilkan MAE/RMSE/R² dan hasil per file; jangan mengklaim R² sebagai persentase prediksi tepat.

In [ ]:
baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)

model = RandomForestRegressor(
    n_estimators=N_TREES,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_train, y_train)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)
pred_base = baseline.predict(X_test)


def metrik(y, pred):
    return {
        'MAE (RPM)': mean_absolute_error(y, pred),
        'RMSE (RPM)': np.sqrt(mean_squared_error(y, pred)),
        'R2': r2_score(y, pred) if len(y) >= 2 and pd.Series(y).nunique() > 1 else np.nan
    }

scores = pd.DataFrame([
    {'Model': 'Baseline (rata-rata)', 'Data': 'Test', **metrik(y_test, pred_base)},
    {'Model': 'Random Forest', 'Data': 'Train', **metrik(y_train, pred_train)},
    {'Model': 'Random Forest', 'Data': 'Test', **metrik(y_test, pred_test)}
])
display(scores.round(4))
scores.to_csv(OUTPUT_DIR / 'metrik_model.csv', index=False)

pred_df = test_data[['recording','window_start_s','window_end_s','setpoint_rpm','rpm_target']].copy()
pred_df['rpm_prediksi'] = pred_test
pred_df['error_rpm'] = pred_df['rpm_prediksi'] - pred_df['rpm_target']
pred_df = pred_df.sort_values(['recording','window_start_s']).reset_index(drop=True)
pred_df.to_csv(OUTPUT_DIR / 'prediksi_test.csv', index=False)
display(pred_df.head(12).round(3))

per_file = (pred_df.groupby('recording')
            .apply(lambda g: pd.Series({
                'n_window': len(g),
                'MAE_RPM': mean_absolute_error(g['rpm_target'], g['rpm_prediksi']),
                'RMSE_RPM': np.sqrt(mean_squared_error(g['rpm_target'], g['rpm_prediksi'])),
            }), include_groups=False).reset_index())
print('Ringkasan error per rekaman:')
display(per_file.round(3))


## 8 — Grafik prediksi laptop dan fitur terpenting

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(y_test, pred_test, alpha=0.75)
lo = min(float(y_test.min()), float(np.min(pred_test)))
hi = max(float(y_test.max()), float(np.max(pred_test)))
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.0, label='Prediksi ideal')
ax.set(xlabel='RPM aktual (encoder)', ylabel='RPM prediksi (MPU6050)',
       title='Validasi regresi pada rekaman terpisah')
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scatter_aktual_prediksi.png', dpi=150)
plt.show()

contoh_file = pred_df['recording'].value_counts().index[0]
s = pred_df[pred_df['recording'] == contoh_file].sort_values('window_start_s')
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(s['window_start_s'], s['rpm_target'], 'o-', label='Encoder (aktual)')
ax.plot(s['window_start_s'], s['rpm_prediksi'], 's--', label='ML (prediksi)')
ax.set(xlabel='Waktu awal window (s)', ylabel='RPM', title=f'Contoh: {contoh_file}')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grafik_satu_rekaman.png', dpi=150)
plt.show()

importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
importance.head(15).iloc[::-1].plot.barh(ax=ax)
ax.set(xlabel='Feature importance (Random Forest)', title='15 fitur getaran paling berpengaruh')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fitur_penting.png', dpi=150)
plt.show()
display(importance.head(15).to_frame('importance').round(4))

## 9 — Latih ulang model FINAL untuk ESP32

Setelah model evaluasi diuji, model final dilatih pada **seluruh window valid**, agar semua rekaman yang tersedia digunakan di perangkat. **Angka metrik di atas bukan akurasi independen model final**, karena model final memiliki akses ke seluruh data; evaluasinya tetap mengacu pada hasil pemisahan sebelumnya.

In [ ]:
model_final = RandomForestRegressor(
    n_estimators=N_TREES,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model_final.fit(features[feature_cols], features['rpm_target'])
bundle = {
    'model': model_final,
    'feature_cols': feature_cols,
    'sample_hz': SAMPLING_HZ,
    'window_n': WINDOW_N,
    'hop_n_training': HOP_N,
    'feature_notes': '36 fitur dari getaran MPU6050 X/Y/Z (AC, FFT), encoder label only',
}
joblib_path = OUTPUT_DIR / 'model_rf_rpm_mpu6050.joblib'
joblib.dump(bundle, joblib_path)
print('Model final:', len(model_final.estimators_), 'pohon,', model_final.n_features_in_, 'fitur')
print('Cadangan Python (.joblib):', joblib_path)


## 10 — Konversi Random Forest menjadi model biner MicroPython

Model scikit-learn `.joblib` **tidak bisa dieksekusi langsung pada MicroPython**. Format biner mandiri `RFR2` menyimpan struktur pohon, threshold dan nilai leaf dengan indeks node 16-bit agar model tidak bergantung pada batas 255 node/pohon. Inferensi di ESP32 memakai pembaca berkas per pohon untuk mengurangi kebutuhan RAM.

In [ ]:
def export_rfr2(model, columns, outpath):
    if model.n_features_in_ != len(columns) or len(columns) > 254:
        raise ValueError('Urutan/jumlah fitur tidak sesuai batas format RFR2')
    if len(model.estimators_) > 65535:
        raise ValueError('Jumlah pohon terlalu besar')
    with open(outpath, 'wb') as out:
        out.write(b'RFR2')
        out.write(struct.pack('<HH', len(model.estimators_), len(columns)))
        for est in model.estimators_:
            tree = est.tree_
            n_nodes = int(tree.node_count)
            if n_nodes < 1 or n_nodes > 65535:
                raise ValueError(f'Pohon mempunyai {n_nodes} node, maksimum 65535')
            out.write(struct.pack('<H', n_nodes))
            for i in range(n_nodes):
                feature = int(tree.feature[i])
                left = int(tree.children_left[i])
                right = int(tree.children_right[i])
                if feature < 0:
                    feature = 255  # leaf
                    left = right = 0
                if left < 0 or right < 0 or left >= 65536 or right >= 65536:
                    raise ValueError('Indeks anak pohon melebihi rentang uint16')
                out.write(struct.pack('<BHHff', feature, left, right,
                         float(tree.threshold[i]), float(tree.value[i, 0, 0])))
    return outpath

bin_path = OUTPUT_DIR / 'model_rf.bin'
export_rfr2(model_final, feature_cols, bin_path)
print('Model biner siap ESP32:', bin_path.name)
print('Ukuran:', round(bin_path.stat().st_size / 1024, 1), 'KiB')
if bin_path.stat().st_size > 1_000_000:
    print('PERINGATAN: model >1 MB; periksa kapasitas flash kosong ESP32 Anda.')


## 11 — Verifikasi kesetiaan hasil konversi

Sebelum file `.bin` diserahkan ke ESP32, kita jalankan mesin pembaca RFR2 **di laptop** dan bandingkan prediksinya dengan scikit-learn. Galat selisih kecil hanya mengukur kesetiaan konversi, **bukan akurasi terhadap RPM aktual**.

In [ ]:
def predict_rfr2_laptop(v, path):
    with open(path, 'rb') as f:
        if f.read(4) != b'RFR2':
            raise ValueError('Magic header RFR2 tidak cocok')
        trees, n_features = struct.unpack('<HH', f.read(4))
        if len(v) != n_features:
            raise ValueError('Jumlah fitur tidak cocok')
        total = 0.0
        for _ in range(trees):
            n_nodes = struct.unpack('<H', f.read(2))[0]
            data = f.read(13 * n_nodes)
            if len(data) != 13 * n_nodes:
                raise RuntimeError('Berkas model terpotong')
            node = 0
            while True:
                feature, left, right, threshold, value = struct.unpack_from(
                    '<BHHff', data, node * 13)
                if feature == 255:
                    total += value
                    break
                node = left if v[feature] <= threshold else right
        return total / trees

X_verify = features[feature_cols].to_numpy(dtype=np.float64)
py_pred = model_final.predict(X_verify)
emb_pred = np.array([predict_rfr2_laptop(row, bin_path) for row in X_verify])
max_difference = float(np.max(np.abs(py_pred - emb_pred)))
print('Jumlah window diverifikasi:', len(X_verify))
print('Maksimum selisih scikit-learn vs RFR2:', f'{max_difference:.8f}', 'RPM')
if max_difference > 0.02:
    raise RuntimeError('Konversi RFR2 tidak cocok. Jangan unggah model!')
print('LOLOS: model biner mereproduksi prediksi Random Forest di laptop.')


## 12 — Bangun paket ZIP dua file untuk ESP32

Sel ini menyertakan program utama MicroPython yang sudah berisi pembacaan MPU6050 (±4g, 200 Hz), fitur FFT, inferensi RFR2, kontrol PID motor, pembacaan encoder, serta perintah setpoint serial. File `main.py` merupakan teks Python biasa yang bisa dibuka dan dipelajari mahasiswa di Thonny. Model biner selalu berasal dari training final pada sel sebelumnya.

Isi ZIP sengaja **hanya dua berkas**: `main.py` dan `model_rf.bin`. Jangan salin `.joblib` ke ESP32.

In [ ]:
# Template main.py disisipkan di dalam notebook agar tidak perlu unduh kode luar.
# Setelah diekstrak, main.py dapat dibuka dan dipelajari seperti biasa.
_MAIN_PY_COMPRESSED_BASE64 = (
    'eNqtPGtz2ziS3/UrsDOVEWVTtCTHHkeJXOXYTuyNX2s7u3fn9aogCZJo8aEhKT+ye/vbr7sBkABJeZK9cSUSCQKNRqNf6G7qp59+Or652u6xc3'
    '+cxFcv2TyO+iwV2TL2owwuEp8HrL3PwjiLE3Z1eoQ311fnTETjeCIS9gtbJmLiL1KfXX/yGo1PfiDYxGcEts9C7kfe8oVNeAQwJiIYJlNv5EfM'
    'mfPUDxiADXnmp9Ah8dnnOJ7B8MM44KOW1/gUJ/CQLQGLKONz9uhzlokk9CNASuLWZ2877Pji9vjaZbv51Z6+anQ7eaO6YM7N7eVVy2U3twe3X2'
    '9kIyB+JFLAtd9grM2uAMFfaLVy3YRcukj8JSATzWariLMXHs0AmwDwApxGq0Xms5FIpqtolvoE5fzq625np8OctzOYLuXhUgSsBwidfHPZkx9N'
    '4ifAvqOfOP/usYnI/EWLRm/vsqmfrRI2gNapH/lAYdmw5BPOojgTozheSGIBNUYiWIXRC5dTI6nZNYc5QgZkFGkGmxIApee4OTTGhatFHD2KBC'
    'CHInrg8MTeo+tP1z2JzbHabug3i6MsiQNJmvdsJjKewO5OfCTLAq4AFz7jQDCerdr4AVgxP1quMnZ+RtBugGwhB86JU5GyOawKaPnp0+2WH01F'
    'IqLUd4n6GcBeEuJAbSLvggdcNiONGWxHOOJjDrMCtaHng5hwj50Qb2VAV75AbvYBiiIu22RPfJGt2CIOASPo6BFKX6NstYAuYRylWQLNbIslPo'
    'xl/IkHLoMNhkkWau2J4EE780PBFgI6e42ffvqpMU2A2CEfz/1IMD9cxkmGnARL+du5y057h0jxlI8CMfST31wQIX0th/Ik4S96IN001A3OpK9n'
    'Y32VvqT5ZZasxpm+A6GZN7LkpU8k0l1EIKCLeB6LZcZOqfE4SeLE6rWS3RhP9QBcWs6L/kRESMRUjGHTC8aE5hmQR/Ki4vs2qgmnwucg141zJU'
    'RIUth2FvrPYtJOgAGf2WqxQnbCAU7vH2/ZBtv5Rw/kJ+PREth+FS5ftlYwjde48YGxAqApaAeRLGFuwISEY5b4E7XNoCaAGUPYp3A18YlP58BP'
    '0QznPjpsKMGhZmJDuVA5eKLwc054FLU89lfQAzQJcB3gACixaRDzrEHabutQKlAa9uBngBRMH6WkQJagyFB4SO1JKQmAcnNAbOwHkoEaFyDssP'
    'DG1WkPrnpeB1aPm+kBt19fXt4ODw8OT47h0T//t3FycIHd7zreDrAvfqq+4zh1EMAGQx52LuBpt9ViU0DDBzFkQN2ZcC5a941GAzaQDZM4zlIn'
    'aklGoDsAXMzngYjDY8kmU9UBVPZFHAk5xhx3NwbBCsSzkyPTltgsAJsINpLaUz+y21s5oPwPMV4UGEeAsX5UIHcX3cOkNLlEX8D2RapBLXA6zY'
    'bEYs4jD1YidZErHv2xGOAS1LojABOISHXJVwvNA9Y1linhy15yRmJb2C7Z/Q1cwJAOE0EqmLOjW3eM1k4BPs4kAGMGDmoHWA0mIPl0msiCo1W2'
    'mgMgJWlT4E8gT29rpylhLXkiyV9Z7d1Dv09T3OfrltzwUNCWniv6hgAlYltbEq+cJ6A15xRqjFcZ8d/DPWxi1KjdsWJV/gRptABChHkb4AdtnY'
    'eGueMVrPoWZ+CYzYFc7t3D/R3AxfkRszvnAdmpBXNEBasAmncL5BEYWWAz1aRARsZNQE5gYC+ZswDTv7vdou2y51ZDnJbJaQBf8xmAGPJnPwXE'
    'n0BhDVfIaWMeiILwEiAIO6hAcGDACCqtH2XdXVATGYe1PwEGqPJTtGAwD9rtKSjJRRKTDfNQWWievaCrCSjNzI8jmBPacOq7CAX/Hj7ornOvWc'
    '7s+sFaY5XzbnMs+GIF6gp0X+ZPwKiCmQTzHynmm+KkDk3Ygi3oil2QaWMeZaoTP56gO2MgsKWHUZef2amhypXqJteJa8aPlp5U9w4sK/KBY6BD'
    '5jLe8giEapOasVOw5gNyWs6kfj2TgicDuk4TEEZvQs8NhXne62mOPu4D+8AQ9R5xDQ14gP5A8Q8Kjs08D8i23byJ61keClYd5W0EqBCLhI+RwA'
    'q9NiwWCTeSV7ClI7YPAKVq8Tr5OEWLOx/ZH5kSJgPIBG5DNUiU20w+bSFgYtnvEhX/O0WFP84Ag3QVOgolnEbuC2qFu0dcCfTBrXnErVHdJAnQ'
    'lVUNdA9w0DzqlcJxYPgI98+ax0P+bDcsBXBsMSCfxVC6BBM26BEI82iu/RG2k2aw16cnLfXdl5OX+ip8Hq3N56PUeWyZwzkMR1Tt0Qp5LpeeoQ'
    'BJC/pbkjmENppPLdzYYR8EsN01NsSPHodyZBdcii3slD8DbyszaLOOPvj3jUn6KHDWMwID9PsGz/P/9iRbA7XlyKf9OgSUDkBzlwoQOjhRgbiD'
    'PfMS+HCeUZiX3hx8MT+ageC23rNl/CQSRU3o0/pHT6qBdClQZAxjmPslzygPGwz9J7iCc2GrqhIMU5mzoDSFq+WSJgQGcN7teB0Xld8GoP92T3'
    'ZFvT57MUj6JPzZPBOT4TQRv5l8C2yh0W93VSOc18r9RihqUp25zPq4rzW6aLp7KNaGYsOtQ5KAJczbltD2zcOTDO6VvNiECzBKM2rBC0MHEVIL'
    'eDJNc+lVXEcP96XHitjSPdgWIpbNRIo8aMKtdptK8Ji+N0q9YK6lEjIiXb/iNZpUXVaeGuTFr99fA+1zlwgud34ThetdySnBLSJzYc5pwEQAZa'
    'Db3wW0ux7odg3Q3e8C2lsPdLcGqGTz3wO6XQX6/3axUOyQbbV2U8zzgRTcTjGyRmrwbyzwtO5PSs0VCbUVkjHKZsotNdL0/u5AEbqMPlBRu0rZ'
    't6WBcEm7uTl+rrVWPZE6EgXxzEH9vKmmabklK2L1USzXcmsfdO+/Y3Rv3ejt+/xwKJ4z8BYyB1ZFTi1/Ud/f5De5qeTo8vF4SJ7DYK/7DoTIrd'
    'tX88ClTj8OHrokeGIIun2xb79Zt3pOarhovea/3sCpn8PZaToF3p3zBDzOIhKhnVfB5cmpUKfoxOOUqFXXLN2YlgB4QCkRTZziEKCB1BKpsC+F'
    'CzAbw3E5wKCLfcIg+LAfP8N66Y+dCzg/szxYxjBO10c3FezhPI5cNhcYOL0+ONdDYLCKpo584K4+DQGhlueOh1WIcUw12G6keIgH429kHI069U'
    'udIjiYvNeRNrzpE9arRPSx454L4wMxzfpymMsSFKz8LpsnIp3HwaRP4ZRtwIzOrvrWa1xcHh0PP53fwj41P3w8OZlOm7Lt43/fHt+gZ7OtOJbC'
    '0UBCNT8QfQruesRDMWia0c1mbRTgyc/mLF4Cl+lRLmsmo2YLI2LTvqUy0WROnLct9qcBGzWRoE1bmVX58SNGMFMZZVUhxancFhz+nolFugT+W2'
    'GASkadZUhtBKzbLDglGmaJwKVFQ71MNPAUBPRW0ZKPF07zw8kJrDHH0nI1UYz0SMLfAATzK/ioaH9vRUfA21Eemobz68riJnVcTEW64r6xgizO'
    '5OG/5HfivEPTFZOYlA0O2m+5sF6r7CDg2ka0qF7VPZALuLxR2Ms4OR054TQ7k9HDOcDVotS0wUdDZO5aYgOtRy194jDQ0UMqtKyn5xVRTTLIIk'
    '4BqRIKcBwnh3nqZSIInNJD/5vA8I2ac4MVMmL1+5nJeSj4yCY+xtBhkxY88Geop+TejUTKE3jKFximR55MV9GsBCgUAaAPDABn9rmK0mKgkwcx'
    'sLqvQp2rZDUnmi2QTRA7r0wowh2cRjQfVUpRaCTfdexbjRiqzceutP/Yqwqphg2OEHho80KJ7LaPUMIKFYjNJwArDzjkncUSlEu5VUYTbpNVDX'
    'gykQBpP9/S15ZjstEFDlMrIgkEfepPmrU0o0UYPlr9HKgfXFLiSnsbalup67JYDDGt4dRCwz+t013CwJVLNVm2im39LhB6XirEwpHSsfk9sEqK'
    '5Pe6Wpql6Nxfu7zv4DFwRED1g4eFSTgeNFt/FOWdgraj2j1HmOg+9XZ26lcgFfTmQMKvJx7QbVF5olgf0dUToVm5w4t7lO8cdxmwohVVBYtkxY'
    'qYmRxLT/d/VChu/EUAHiAp+VwXLQuF26zYEi0a9WJR4rmqUnrtDKQs3S+s9hy0Ntws92VLm2hyC9v2H6Wvp5QkJxdissJUK19k/mIVljs3rk4v'
    'hscXB5jA2GG2aj/rvdu7YPjQAZuO7uUEdNWSpy0adXrRxVG/1o6Ch6oTpbJ213TqqfkPgV+vh4hFd0/3Oafcuy40OOxaXT9i13druvaYM/GXfj'
    'QBN0QWFlydXva6Eu2bI1pst4SRztbDY9nt8Ay79dZ1OzyrI/0XMznObo+vv/75VKKQFw4YpL+6him2tzuNL1cyONf4coqTdrztxpcj5R6dH/wX'
    'UPH2+PP1AWLU7XSoFVC8+ts59e404Gr46fr4L+p5ge7Jt8bh5cXt9eXZ8Or4+vTyaHiO3vKOBHtzfHt1CbBx2E5Hj4GNpvxoJpPbeB8hIf3Vg0'
    '9HJ8zt6xIRP/Lr6IDZ+F+Uumu//te4OTi/OjsenvyPWoy6V/h+vVGLwnVtbbGiN6C6w8K0cXB4eHw2vD64+Hw8/IwZ1Hz1fwb3QyFBBQchHIsW'
    'gP3ezFW1BXkBxEKwGUKCfT84wxSrPMWiCJfg4wSkud523u3iTgA/ng0/ndIo+4jRuPx6e/X1dohd8OHtwcez46bETd+EK+JScsHes+bhzV+bbE'
    'XFCHAqnwE3B3yZxUsksuRysB5zzMJL40HKZ7X0GuA9P/FkMgS2h6lABThKRl288QATZSsGnVYDJlvU9u7V9VaFCssnTKsAqzm6P6iGlkuhooHm'
    'QLO3N1llLw4AgLbxkBsT5QIvpzu9aFGXUU2Xj1YXlO+hNnmdBvic2VCLfd4wlkI4DCkXBgcUL/PHixTuQZdi9miW5IcPCQHtg2pIliFQRceMYB'
    'VDlaZsQOswFb+tYD45ey4EeENsFEfqAdYmDVXcATa+qcMpagV+8psDpFfHmlkQj2AOa3kqFn6FUFMrA7aQrh2yAtUgpcRAI5E8cHDRvSJWBiT3'
    'CJbTAltJ9NX3fSMYZtI0N7i2o2V3amMnez3jGFhWg4XFDYEtMySSUfCi7JjevOpqi3oYJwdh2T7ZVc6cgkwMiQqOTUW1YyZEkxXVQVMzvyKIem'
    'CKhf3E5AOJADgxIkNOKc1vsqTLSvzoMs1+6hFxHrh1kukITomnSyQuurzK5TLtY3F6AdvgdqKuxfEqioK8nonlH7g4VxPRZaYk5ZmrXJjsnDRt'
    'vcx2x09rVgoqN+MVQkz86dSBQRVE82xZPu4Dq1jJWhQmGZXjqVFbyhhboPZ1kUdXtHfX7BYgRU9oT9fs8QTzM7JD26J7HZNQv9JmOgBhi4GPgS'
    'l5ygxgUj5T/GzqMpJ6KTCKNfLNaFusmXMUjJBdNwBimdkwZxDyZ6fghrbpw4DFsG7VDib+I8/8R1QNjoTdNrioRbjrupMllZ6A07Sh0Nhk4Dht'
    'GPgxcJ82DKhV7qfvRkm4DdQdTCnKuVotzBK6sNu97Zx38LFCbZ91yafXcPZNBjaAA+BcBJT/1qrVU6qXLoo4SEDByxCa1P7gAoAv1kcne9Bx0Y'
    '0edKU7oIizjFM/86e5MVDN+5ZorVWDa1ShKtKwTUMVSPdVIOgNqDgtEHmY0cJlxE3rGnriNHVhk7xj7Pbg+vPxLfsXU54BXF1dHx+dfrk5hUt2'
    'c3x2enN6gpfoF/+LnZzefr34XIainM7N3P0srmou85amRhtOrk9D0P2rIFP1GC7LFaKKPQvQdaQPSOW5bEylpoJU5DRFdZSJfNQwFDwqUiKWxz'
    'iQ/qDBTHIZ18c3X89u3TdedwofPf2xSZ8T+kePJvSgyd6Uzpb/Cebl4+mahdTxSE78N+AvT2Fr3vwKaOH3nvre1BdvJ+ozTP8QtFs5u/EkFaBg'
    'wxAk1RmHE9uwaYUHXq/046R+DlGRwqeXZokPxtCjLKhjle/B41pbAY9xPKaQmrLWvOmy5k2zVSGMfOrK1Q1g1zRFQe8OaGthVdgO5xfMIwq4rq'
    'FOsYaqxZ2NPT3YaRm5AAPfvGBYmRFYOWKHpJJ7KiuHi4BKZR1XulBfhhwn/gIUW+CxL1R4/baDJfpYnI+aFCsYdA1+s7WGfkjeDsaMEB34Ms+s'
    'VSpqmyXzfB3PezPBVxWAUtbAdZPRkgvTtx6+PDkWwGFgLUzjeABdpPOrDgm5n226scrVzd3KH1IJenUusUaBU60k/j3KF4OZpf66xdTpaIo4pd'
    'kE+JpPuEz0JBn7evOxrV4XuT6+OpNBtpC/xImf8VS+DsKOxOMXP/P0+WgZBwE5G5jz9PAOwBvPvETMfPBCEyd9ST2a09W9ry7PzvBMKKV7PBfj'
    'xVAOdcqSbZzFlEH9iOmOG2A9tuAYGMNsD2V8gsBVzLukQAFGhgM/whPUi0zJYR33KIjHCz+aeTk4fI9A571skAL0zWyGFedwUFvyFy5fbKA5oF'
    'O2iibcyxPPwyLz1XvbstKNKAsmbYhcnXJizArJjjHbkJNOhrm7rTLQ8fw1GKjF5sR2f0+amBvUd1Ep2SljnQWpq/FMWwVbnVt1wc/yEbpIAhhI'
    'jSyknn+dNisFJRYc6/6u3zbKLAkuxvht1OBw0Ou8BhSEeTwHRhyDf5myC/Gkhcs5JpUJAq89G+ig/QgV3BXPS4EGbKhUg5mMtrm68NvpXJaf80'
    'sAKpFm3f6nQaVvuXTCxF2Llh/52RDMaX6672Fh3Wnv0AH9nU74QAdsbo4wGJSOg6Ll8EyHh95S/E4fv1dU0QaQvHTMo2ItnefdPZQA6lEgxycT'
    'Oq/C00a+UXD37tW+72r0Xzn9q4K52l5lIsScvMs4hgszXOXgDToicpaWJoD3BFpNZDG4PKGDM7o44UeXjYAFOx1lz+gomgZCLPGk2tXLXze+e6'
    'DG95ot1CpHZ1efBr3Xh7xTQ97KITccSxxhlZnASOrJt9dHH7r1eR2EuNd8LfSJXbp6nSo6A9O4RH7FOHz828pPhJO3g1+2ShIBFkWxX1ENfxCO'
    '/MAozdHhWQppYQoZlSZsVJhKXXlJ55k+1cpTAX3qFhXzcG2+oQjU4G38UO/cyXQjds9VuK6n51iNTPCc5hy8tTsqIL9QRcwvrz389spDKi9SD0'
    '8rD/3xomMHLlY6rhGJZzC/0EbPoV8RNeisCRug65cSGxg1TXTwLYUvKzEYrFwLZNSuU1dYalZaae1V3tBSjXwlGKPX45aX27IPzd81R0nGVoaM'
    '6VpRc8mg/yp0KFW3FLTzODj6YKXwlK+86dIrSt9FU1WwES7XbLDkDvl+SplYNM6V+170flSKE405ptctid4GFbRb9OXPABkL1+TXNzlPqXJlfz'
    '5HhjVK0W2uy5GCScwNLGdpivHIRJFI66JxJRq4xVx2eZKGsJ9zZbWkXvGq7ls/flDB0wb0yloruNavmISvuy6Ip191oRdkYKfb8o0cfLkEQ1pd'
    'sVtE7VagWekQrerZFeS2EngM46konoOZww0Duo4UnfGRCFTNmPLOQ4GvPeaMzMfZigcgb6l8B1fqRO3JflYvZcpBxtu1Uv+GqUpNgWuLlaI+JS'
    'RAVALRzuI2XXjGeUpLk3WuzEUshXODIQchRS3R/zI6taia3Rqu3ijFN1qKfncw+l7XaFjj3xjDq8UkFSDypalNVoaNu9WrO1JZKNn7aFpH/oyC'
    'iFIIIp26xjC3NMrNuVsZUnyHXntfP7Mr0DtUVYUFirKaUx1LdKGWfqVV5ohGIsH3nBOvVN9YpC3rKxxVFVxeH7NXKSCUPageZg+dcHl/1397/w'
    'MVkaoq1ayIxHGyklAukRYDE84WfPmfFUEqzN727+3TT1HmKKsecziA//bu76F+bpRXyehsX1aTYKhGFkTWBmpKf07NQlpW1FJu1eUXl4p2qTNO'
    'oTvLSV6Dk7tgFNPR/rwKPWOaEHNuWeJjvnlA2dbrvwyvT29OLz4D8UDQA2g3cpcWdvJov5n7W5t5jcRm6RcIrsUMvtGI21FZ6D/oPL/p9J7fS1'
    '9jsLnVfjOZvSdvEM7fM1jiybf37KJCT2X6LD/VLWoEXPCyrKlgcX2saBn0dih63vuVoue9XfyFg4NBdw+/Pw6677AOZNDrYp3HoNez8ZXxrDy8'
    '44R+ytF7lUy7i6mCY3xH8T3rDPDHJnSca+BH09gGdZrXcKvi6X+Xf6cAfxbh+tN74/cQ8lRv88dCRF/iIA776uVEt4jaoqp3w8ClbMYQLjCGak'
    'R+p+lw/s1FhYQ17Ga0t/lKjEkJdoYWqZ9H5wfkmuc/GdI2PfVmqz4wqmwOkIX7AdBU3dPb9PRzBPTbIqDrNEHSIH4aPvEkEmhRPnFArxrhXFd3'
    'aVddWdENtd8faqu17NNeb6ca0sAUoB+ViulK3i2+YmcEoHPgJuZr3xcw/8r2Rm8b2Bo7Yu+azpQ+s9VXQK4/yFXr977T8bEzCthX/l6B+f5wWq'
    'V1Z00i2NrIR/IOw9GE97/vMKGLLZUpKd59sUmZVwm5MEcVws+MfsFihj+qEsb4wzECWBV/JCW3zFRuTcaN7F5N9WNQ2sAf23x0KEIMrVZfhjDN'
    'fh324BUuUzGpT6aXSA6IdVp1yGMmPFDuEFy0Te6r7pORUStSGBa/SjhuDtp91aLmSY9iLTLvphm9lLKqJb+m2I8RHuNxBQH3sapOvk5H0dtCLf'
    'XXlRiXFbksEltfZJyn1sCQxPQKkPwZE/LhQ5EE5FHtd90daVbeM/m7S+TlL4AZeQKu1cifswf8FZ+Zt6YGeX3htYHF4QHo64MLGw2UQJ69p9x8'
    'golwicD+tqdQWjOjrcRRT9v1+DIbZUQs+zWSeCiDzUyZZvUCvcx58BEn06Gcd4rvS/HEFEPKfe/H9t5S71R+GxhKu5LosZ0v7WRR9LdsBs+1I0'
    '/mjlLRuUGQhwP9sz5fxMso5smEfkYhWS0zK6N+5Of2Uv9gT5YEm4cA8f8A/j6ryQ=='
)
MAIN_PY = zlib.decompress(base64.b64decode(_MAIN_PY_COMPRESSED_BASE64)).decode('utf-8')
assert "b'RFR2'" in MAIN_PY and "def extract" in MAIN_PY
main_path = OUTPUT_DIR / 'main.py'
main_path.write_text(MAIN_PY, encoding='utf-8')
compile(MAIN_PY, 'main.py', 'exec')

zip_path = BASE_DIR / 'ESP32_RPM_MPU6050_Siap_Unggah.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(main_path, arcname='main.py')
    z.write(bin_path, arcname='model_rf.bin')
with zipfile.ZipFile(zip_path) as z:
    assert sorted(z.namelist()) == ['main.py', 'model_rf.bin']
print('ZIP berhasil dibuat:', zip_path)
print('Isi:')
with zipfile.ZipFile(zip_path) as z:
    for i in z.infolist():
        print(' -', i.filename, ':', round(i.file_size/1024,1), 'KiB')

# Simpan laporan metrik terpisah pada sesi Colab (bukan isi ZIP ESP32).
report_path = OUTPUT_DIR / 'ringkasan_evaluasi.json'
report_path.write_text(json.dumps({
    'n_file': int(features['recording'].nunique()),
    'n_window': int(len(features)),
    'n_features': len(feature_cols),
    'split_method': split_method,
    'n_train': len(train_data),
    'n_test': len(test_data),
    'mae_test': float(mean_absolute_error(y_test, pred_test)),
    'rmse_test': float(np.sqrt(mean_squared_error(y_test, pred_test))),
    'r2_test': float(r2_score(y_test, pred_test)) if len(y_test)>1 and y_test.nunique()>1 else None,
    'max_conversion_difference_rpm': max_difference,
}, indent=2), encoding='utf-8')

if IS_COLAB:
    colab_files.download(str(zip_path))


## 13 — Petunjuk deployment mahasiswa

1. Ekstrak `ESP32_RPM_MPU6050_Siap_Unggah.zip` hasil sel 12. **Unggah `main.py` dan `model_rf.bin`** ke root ESP32 memakai *Thonny → Files*. Jangan hanya membuka file di komputer; pastikan keduanya tersimpan pada perangkat.
2. Pastikan koneksi: **ENA 25; IN1 27; IN2 26; encoder A 18; encoder B 19; MPU6050 SDA 21 dan SCL 22; VCC MPU 3V3; GND bersama**. Lepas jumper ENA L298N.
3. Reboot ESP32 atau jalankan `main.py` melalui Thonny. Karena file bernama `main.py`, ia dapat **berjalan otomatis saat boot**; hentikan dengan Ctrl+C jika perlu. Untuk mode manual, ubah nama `main.py` di ESP32 lalu jalankan dari Thonny.
4. Masukkan angka **40** lalu Enter untuk target 40 RPM; lanjut **60**, **80**, atau **100**. Perintah `0` = stop dan `STATUS` = ringkasan.
5. Amati tabel **TARGET, ENCODER, PREDIKSI, SELISIH, PWM, HITUNG**. `SELISIH` = RPM prediksi − RPM encoder. Jika perlu hasil yang dapat diproses otomatis, ubah `OUTPUT_MODE = 'CSV'` dalam `main.py`. RPM encoder adalah label pembanding, **tidak masuk fitur model**.
6. Jika serial tidak menerima input ketika skrip dijalankan via Thonny, coba terminal serial terpisah 115200 baud dan jalankan `main.py` dari REPL. Hanya satu aplikasi yang boleh membuka port serial sekaligus.

**Peringatan praktik:** kontrol PID memakai nilai `Kp=1`, `Ki=20.3`, `Kd=0`, `MIN_PWM=200`, `PPR=330` mengikuti rangkaian motor yang sudah bekerja. Ketika output PID negatif, kode asal membalik arah motor. Awasi motor saat uji, dan gunakan `0` untuk menghentikannya.

**Batasan evaluasi:** model hanya berlaku pada kondisi motor, orientasi MPU6050, rentang RPM, dan pola beban yang menyerupai data latih. FPGA/MCU compute time belum diverifikasi pada ESP32 fisik; bila `compute_ms` terlalu besar atau muncul `MemoryError`, optimasi fitur/FFT atau jumlah pohon perlu dilakukan pada iterasi berikutnya.

**Keberlanjutan praktikum:** untuk melatih model baru, cukup ulangi notebook ini dengan dataset yang baru. Paket ESP32 akan dibuat ulang otomatis, dan mahasiswa tetap hanya perlu menyalin dua file.